In [8]:
# Install dependencies
!pip install transformers torch faiss-cpu sentence-transformers PyPDF2 accelerate -q

# Upload document
from google.colab import files
uploaded = files.upload()

import os
filename = list(uploaded.keys())[0]

# Load & chunk document
def load_document(filepath):
    if filepath.endswith(".pdf"):
        import PyPDF2
        text = ""
        with open(filepath, "rb") as f:
            r = PyPDF2.PdfReader(f)
            for page in r.pages:
                text += page.extract_text() + "\n"
        return text
    else:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

def split_into_passages(text, passage_length=200, overlap=50):
    words = text.split()
    passages, start = [], 0
    while start < len(words):
        passages.append(" ".join(words[start:start + passage_length]))
        start += passage_length - overlap
    return passages

raw_text = load_document(filename)
passages = split_into_passages(raw_text)

# load models
from transformers import (
    DPRQuestionEncoder, DPRQuestionEncoderTokenizer,
    DPRContextEncoder, DPRContextEncoderTokenizer,
    pipeline as hf_pipeline,
    AutoTokenizer, AutoModelForSeq2SeqLM,
)
from sentence_transformers import CrossEncoder
import torch, numpy as np, faiss, warnings, logging, re
from collections import Counter

warnings.filterwarnings("ignore")
logging.set_verbosity_error() if hasattr(logging, "set_verbosity_error") else None

import transformers
transformers.logging.set_verbosity_error()

q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
q_encoder   = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
c_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
c_encoder   = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
q_encoder.eval(); c_encoder.eval()

reranker          = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
extractive_reader = hf_pipeline("question-answering", model="deepset/roberta-base-squad2")

gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
gen_model     = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
gen_model.eval()

# faiss
all_embs = []
for i in range(0, len(passages), 32):
    batch  = passages[i:i+32]
    inputs = c_tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        embs = c_encoder(**inputs).pooler_output.numpy()
    all_embs.append(embs)

passage_embs = np.vstack(all_embs)
faiss.normalize_L2(passage_embs)
index = faiss.IndexFlatIP(passage_embs.shape[1])
index.add(passage_embs)

def normalize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

def exact_match(predicted, ground_truth):
    return int(normalize(predicted) == normalize(ground_truth))

def f1_score_tokens(predicted, ground_truth):
    pred_tokens  = normalize(predicted)
    truth_tokens = normalize(ground_truth)
    common       = Counter(pred_tokens) & Counter(truth_tokens)
    num_common   = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)

def passage_contains_answer(passage, ground_truth):
    return ground_truth.lower() in passage.lower()

def evaluate(question, ground_truth, k=5):
    # Retrieve
    inputs = q_tokenizer(question, return_tensors="pt", truncation=True, max_length=64)
    with torch.no_grad():
        q_emb = q_encoder(**inputs).pooler_output.numpy()
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)
    top_k_passages  = [passages[i] for i in indices[0]]

    # Recall@k
    recall_at_k = 1.0 if any(passage_contains_answer(p, ground_truth) for p in top_k_passages) else 0.0

    # Rerank
    rscores      = reranker.predict([(question, p) for p in top_k_passages])
    best_passage = sorted(zip(top_k_passages, rscores), key=lambda x: x[1], reverse=True)[0][0]

    # Predict
    predicted = extractive_reader(question=question, context=best_passage)["answer"]

    return {
        "recall_at_k": recall_at_k,
        "em":          exact_match(predicted, ground_truth),
        "f1":          f1_score_tokens(predicted, ground_truth),
    }

eval_set = [
    ("When was APJ Abdul Kalam born?",         "15 October 1931"),
    ("Where was Abdul Kalam born?",            "Rameswaram"),
    ("What award did Kalam receive in 1997?",  "Bharat Ratna"),
    ("When did Abdul Kalam become President?", "2002"),
    ("Which missile did Kalam develop?",       "Agni"),
]

results    = [evaluate(q, gt) for q, gt in eval_set]
avg_recall = sum(r["recall_at_k"] for r in results) / len(results)
avg_em     = sum(r["em"]          for r in results) / len(results)
avg_f1     = sum(r["f1"]          for r in results) / len(results)

print("=" * 35)
print(f"  Recall@5      : {avg_recall:.2f}")
print(f"  Exact Match   : {avg_em:.2f}")
print(f"  F1 Score      : {avg_f1:.2f}")
print("=" * 35)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.1 MB/s eta 0:00:00


Saving A._P._J._Abdul_Kalam.pdf to A._P._J._Abdul_Kalam.pdf


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

  Recall@5      : 0.80
  Exact Match   : 0.20
  F1 Score      : 0.24


In [12]:
# ============================================================
# CELL 9 — Ask multiple questions (wider generative context)
# ============================================================
def ask(question, k=5):
    inputs = q_tokenizer(question, return_tensors="pt", truncation=True, max_length=64)
    with torch.no_grad():
        q_emb = q_encoder(**inputs).pooler_output.numpy()
    faiss.normalize_L2(q_emb)
    _, indices   = index.search(q_emb, k)
    top_k        = [passages[i] for i in indices[0]]

    rscores      = reranker.predict([(question, p) for p in top_k])
    ranked       = sorted(zip(top_k, rscores), key=lambda x: x[1], reverse=True)
    best_passage = ranked[0][0]

    extractive = extractive_reader(question=question, context=best_passage)["answer"]

    # Use top-3 passages as context for richer generative answer
    top3_context = " ".join([p for p, _ in ranked[:3]])
    prompt  = f"Answer the question in detail using the context below.\n\nContext: {top3_context}\n\nQuestion: {question}\n\nAnswer:"
    inputs  = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    with torch.no_grad():
        output = gen_model.generate(**inputs, max_new_tokens=300, num_beams=4, min_new_tokens=50)
    generative = gen_tokenizer.decode(output[0], skip_special_tokens=True)

    print(f"Question   : {question}")
    print(f"Extractive : {extractive}")
    print(f"Generative : {generative}")
    print("-" * 60)

questions = [
    "What was Abdul Kalam's childhood like in Rameswaram?",
    "How did Kalam contribute to India's space research at ISRO?",
    "What is the Agni missile and what was Kalam's role in it?",
    "What books did APJ Abdul Kalam write during his lifetime?",
    "Why is Abdul Kalam called the Missile Man of India?",
]

for question in questions:
    ask(question)

Question   : What was Abdul Kalam's childhood like in Rameswaram?
Extractive : youngest of five siblings
Generative : As a young boy, he delivered newspapers to support the family's meager income. As a young boy, he delivered newspapers to support the family's meager income. As a young boy, he delivered newspapers to support the family's meager income
------------------------------------------------------------
Question   : How did Kalam contribute to India's space research at ISRO?
Extractive : intimately involved in India's civilian space programme and military missile development efforts
Generative : Kalam played a pivotal organisational, technical, and political role in Pokhran-II nuclear tests in 1998, India's second such test after the first test in 1974. Kalam also played a pivotal organisational, technical, and political role in Pokhran-II nuclear tests
------------------------------------------------------------
Question   : What is the Agni missile and what was Kalam's role i